# Session 11: Advanced Retrieval with LangChain

## Learning Objectives:

- Understand and implement multiple retrieval strategies for RAG
- Compare naive, BM25, multi-query, parent-document, contextual compression, ensemble, and semantic chunking approaches
- Build RAG chains over a health and wellness knowledge base using LangChain and QDrant

In the following notebook, we'll explore various methods of advanced retrieval using LangChain!

We'll touch on:

- Naive Retrieval
- Best-Matching 25 (BM25)
- Multi-Query Retrieval
- Parent-Document Retrieval
- Contextual Compression (a.k.a. Rerank)
- Ensemble Retrieval
- Semantic chunking

We'll also discuss how these methods impact performance on our set of documents with a simple RAG chain.

There will be two breakout rooms:

- 🤝 Breakout Room Part #1
  - Task 1: Getting Dependencies!
  - Task 2: Data Collection and Preparation
  - Task 3: Setting Up QDrant!
  - Task 4-10: Retrieval Strategies
- 🤝 Breakout Room Part #2
  - Activity: Evaluate with Ragas

---

# 🤝 Breakout Room Part #1

## Task 1: Getting Dependencies!

We're going to need a few specific LangChain community packages, like OpenAI (for our [LLM](https://platform.openai.com/docs/models) and [Embedding Model](https://platform.openai.com/docs/guides/embeddings)) and Cohere (for our [Reranker](https://cohere.com/rerank)).

We'll also provide our OpenAI key, as well as our Cohere API key.

> NOTE: Create a `.env` file in this directory with `OPENAI_API_KEY` and `COHERE_API_KEY` to avoid being prompted each time.

In [1]:
import os
import getpass
from dotenv import load_dotenv

load_dotenv()

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API Key:")

In [2]:
if not os.environ.get("COHERE_API_KEY"):
    os.environ["COHERE_API_KEY"] = getpass.getpass("Cohere API Key:")

## Task 2: Data Collection and Preparation

We'll be using our Health and Wellness Guide - a comprehensive resource covering exercise, nutrition, sleep, stress management, habits, and common health concerns.

### Data Preparation

We'll load the wellness guide as a single document, then split it into smaller chunks using a `RecursiveCharacterTextSplitter` for our vector store. We also keep the raw (unsplit) document for use with the Parent Document Retriever and Semantic Chunker later.

In [3]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader = TextLoader("data/HealthWellnessGuide.txt")
raw_docs = loader.load()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
wellness_docs = text_splitter.split_documents(raw_docs)

Let's verify our data was loaded and split correctly!

In [4]:
print(f"Raw documents: {len(raw_docs)}")
print(f"Split chunks: {len(wellness_docs)}")
print(f"\nExample chunk:\n{wellness_docs[0]}")

Raw documents: 1
Split chunks: 45

Example chunk:
page_content='The Personal Wellness Guide
A Comprehensive Resource for Health and Well-being

PART 1: EXERCISE AND MOVEMENT

Chapter 1: Understanding Exercise Basics

Exercise is one of the most important things you can do for your health. Regular physical activity can improve your brain health, help manage weight, reduce the risk of disease, strengthen bones and muscles, and improve your ability to do everyday activities.' metadata={'source': 'data/HealthWellnessGuide.txt'}


## Task 3: Setting up QDrant!

Now that we have our documents, let's create a QDrant VectorStore with the collection name "wellness_guide".

We'll leverage OpenAI's [`text-embedding-3-small`](https://openai.com/blog/new-embedding-models-and-api-updates) because it's a very powerful (and low-cost) embedding model.

> NOTE: We'll be creating additional vectorstores where necessary, but this pattern is still extremely useful.

In [5]:
from langchain_qdrant import QdrantVectorStore
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

vectorstore = QdrantVectorStore.from_documents(
    wellness_docs,
    embeddings,
    location=":memory:",
    collection_name="wellness_guide",
)

## Task 4: Naive RAG Chain

Since we're focusing on the "R" in RAG today - we'll create our Retriever first.

### R - Retrieval

This naive retriever will simply look at each review as a document, and use cosine-similarity to fetch the 10 most relevant documents.

> NOTE: We're choosing `10` as our `k` here to provide enough documents for our reranking process later

In [6]:
naive_retriever = vectorstore.as_retriever(search_kwargs={"k" : 10})

### A - Augmented

We're going to go with a standard prompt for our simple RAG chain today! Nothing fancy here, we want this to mostly be about the Retrieval process.

In [7]:
from langchain_core.prompts import ChatPromptTemplate

RAG_TEMPLATE = """\
You are a helpful and kind assistant. Use the context provided below to answer the question.

If you do not know the answer, or are unsure, say you don't know.

Query:
{question}

Context:
{context}
"""

rag_prompt = ChatPromptTemplate.from_template(RAG_TEMPLATE)

### G - Generation

We're going to leverage `gpt-4.1-nano` as our LLM today, as - again - we want this to largely be about the Retrieval process.

In [8]:
from langchain_openai import ChatOpenAI

chat_model = ChatOpenAI(model="gpt-4.1-nano")

### LCEL RAG Chain

We're going to use LCEL to construct our chain.

> NOTE: This chain will be exactly the same across the various examples with the exception of our Retriever!

In [9]:
from langchain_core.runnables import RunnablePassthrough
from operator import itemgetter
from langchain_core.output_parsers import StrOutputParser

naive_retrieval_chain = (
    # INVOKE CHAIN WITH: {"question" : "<<SOME USER QUESTION>>"}
    # "question" : populated by getting the value of the "question" key
    # "context"  : populated by getting the value of the "question" key and chaining it into the base_retriever
    {"context": itemgetter("question") | naive_retriever, "question": itemgetter("question")}
    # "context"  : is assigned to a RunnablePassthrough object (will not be called or considered in the next step)
    #              by getting the value of the "context" key from the previous step
    | RunnablePassthrough.assign(context=itemgetter("context"))
    # "response" : the "context" and "question" values are used to format our prompt object and then piped
    #              into the LLM and stored in a key called "response"
    # "context"  : populated by getting the value of the "context" key from the previous step
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's see how this simple chain does on a few different prompts.

> NOTE: You might think that we've cherry picked prompts that showcase the individual skill of each of the retrieval strategies - you'd be correct!

In [10]:
naive_retrieval_chain.invoke({"question" : "What exercises can help with lower back pain?"})["response"].content

'Exercises that can help with lower back pain include:\n\n- **Cat-Cow Stretch:** Start on your hands and knees, alternate between arching your back up (cat) and letting it sag down (cow). Do 10-15 repetitions.\n\n- **Bird Dog:** From hands and knees, extend opposite arm and leg while keeping your core engaged. Hold each for 5 seconds, then switch sides. Do 10 repetitions per side.\n\n- **Pelvic Tilts:** Lie on your back with knees bent, flatten your back against the floor by tightening your abs and tilting your pelvis up slightly. Hold for 10 seconds and repeat 8-12 times.\n\n- **Partial Crunches:** Lie on your back with knees bent, cross arms over your chest, tighten stomach muscles, and raise shoulders off the floor. Hold briefly, then lower. Do 8-12 repetitions.\n\n- **Knee-to-Chest Stretch:** Lie on your back, pull one knee toward your chest while keeping the other foot flat. Hold for 15-30 seconds, then switch legs.\n\nThese exercises are gentle stretches and strengthening movemen

In [11]:
naive_retrieval_chain.invoke({"question" : "How does sleep affect overall health?"})["response"].content

'Sleep plays a vital role in maintaining overall health. It is essential for physical repair, immune function, mental well-being, and cognitive processes like memory consolidation. During sleep, the body heals tissues, regulates hormones related to growth and appetite, and supports brain functions. Adequate sleep—typically 7 to 9 hours for adults—also helps strengthen the immune system, manage stress effectively, and promote emotional stability. Poor sleep or sleep disorders like insomnia can negatively impact these health aspects, leading to issues such as weakened immunity, mental health challenges, and impaired physical recovery. Therefore, good sleep hygiene and a restful environment are important strategies for supporting overall health and wellness.'

In [12]:
naive_retrieval_chain.invoke({"question" : "What are some natural remedies for stress and headaches?"})["response"].content

'Some natural remedies for stress and headaches include:\n\n- Drinking water to stay hydrated\n- Applying cold or warm compresses to the head or neck\n- Resting in a dark, quiet room\n- Gentle massage of the temples and neck\n- Using essential oils such as peppermint or lavender\n- Engaging in deep breathing exercises, like inhaling for 4 counts, holding, and exhaling\n- Practicing progressive muscle relaxation by tensing and releasing muscle groups\n- Taking short walks, especially in nature\n- Listening to calming music\n- Maintaining a regular sleep schedule and establishing a calming evening routine\n\nThese techniques can help reduce tension, manage stress levels, and alleviate headache symptoms naturally.'

Overall, this is not bad! Let's see if we can make it better!

## Task 5: Best-Matching 25 (BM25) Retriever

Taking a step back in time - [BM25](https://www.nowpublishers.com/article/Details/INR-019) is based on [Bag-Of-Words](https://en.wikipedia.org/wiki/Bag-of-words_model) which is a sparse representation of text.

In essence, it's a way to compare how similar two pieces of text are based on the words they both contain.

This retriever is very straightforward to set-up! Let's see it happen down below!


In [13]:
from langchain_community.retrievers import BM25Retriever

bm25_retriever = BM25Retriever.from_documents(wellness_docs)

We'll construct the same chain - only changing the retriever.

In [14]:
bm25_retrieval_chain = (
    {"context": itemgetter("question") | bm25_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's look at the responses!

In [15]:
bm25_retrieval_chain.invoke({"question" : "What exercises can help with lower back pain?"})["response"].content

'Exercises that can help with lower back pain include:\n\n- Cat-Cow Stretch: Start on hands and knees, alternate between arching your back up (cat) and letting it sag down (cow). Do 10-15 repetitions.\n- Bird Dog: From hands and knees, extend opposite arm and leg while keeping your core engaged. Hold for 5 seconds, then switch sides. Do 10 repetitions per side.\n- Pelvic Tilts: Lie on your back with knees bent, flatten your back against the floor by tightening your abs and tilting your pelvis up slightly. Hold for 10 seconds and repeat 8-12 times.\n\nThese gentle stretching and strengthening exercises can help alleviate lower back discomfort and prevent future episodes.'

In [16]:
bm25_retrieval_chain.invoke({"question" : "How does sleep affect overall health?"})["response"].content

'Sleep has a significant impact on overall health. Maintaining a consistent sleep schedule and creating an optimal sleep environment—such as keeping the room cool, dark, quiet, and comfortable—are essential practices. Good sleep hygiene, including relaxing bedtime routines, limiting screen time before bed, and avoiding caffeine late in the day, can improve sleep quality. Proper sleep supports immune function, mental health, nutrient absorption, and overall wellness. Conversely, issues like insomnia can disrupt these benefits and negatively affect health.'

In [17]:
bm25_retrieval_chain.invoke({"question" : "What are some natural remedies for stress and headaches?"})["response"].content

'Some natural remedies for stress include relaxation techniques such as deep breathing exercises, meditation, progressive muscle relaxation, and herbal teas like chamomile or valerian root. For headaches, staying well-hydrated, managing stress effectively, ensuring adequate sleep, eating regular balanced meals, and reducing eye strain can help prevent or alleviate symptoms. Exploring these options may help manage stress and headaches naturally.'

It's not clear that this is better or worse, if only we had a way to test this (SPOILERS: We do, the second half of the notebook will cover this)

### ❓ Question #1:

Give an example query where BM25 is better than embeddings and justify your answer.

##### Answer: 

BM25 is a keyword-based sparse retrieval method that ranks documents based on exact term matches and its match frequency. For the generic queries, both naive embedding-based retriever and BM25 perform equally, it is unclear to comment about better or worse in those cases.

 I tested it with specific query below: 
 
 "What does the guide recommend for managing sugar levels?". 
 
 BM25 performed better than embedding-based retriever. This is because the query contain a specific phrase and target a narrow information. Here, the wellness guide explicitly includes phrase "sugar levels", BM25 retriever prioritize document chunks containing that phrase. BM25 excels when exact words matters, where precision is more important than semantic generalization. 



In [18]:
naive_retrieval_chain.invoke({"question" : "What does the guide recommend for managing sugar levels?"})["response"].content

'The guide recommends managing sugar levels by including nutrient-rich foods such as fruits, vegetables, whole grains, and nuts in your diet. It emphasizes eating a balanced diet with complex carbohydrates, which can help regulate blood sugar. Additionally, it suggests practicing mindful eating by eating slowly and chewing thoroughly, and avoiding processed foods and artificial sweeteners. Staying hydrated and maintaining a healthy overall lifestyle are also important for managing sugar levels.'

In [19]:
bm25_retrieval_chain.invoke({"question" : "What does the guide recommend for managing sugar levels?"})["response"].content

'The provided guide does not specifically mention recommendations for managing sugar levels.'

## Task 6: Contextual Compression (Using Reranking)

Contextual Compression is a fairly straightforward idea: We want to "compress" our retrieved context into just the most useful bits.

There are a few ways we can achieve this - but we're going to look at a specific example called reranking.

The basic idea here is this:

- We retrieve lots of documents that are very likely related to our query vector
- We "compress" those documents into a smaller set of *more* related documents using a reranking algorithm.

We'll be leveraging Cohere's Rerank model for our reranker today!

All we need to do is the following:

- Create a basic retriever
- Create a compressor (reranker, in this case)

That's it!

Let's see it in the code below!

In [20]:
from langchain.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain_cohere import CohereRerank

compressor = CohereRerank(model="rerank-v3.5")
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor, base_retriever=naive_retriever
)

Let's create our chain again, and see how this does!

In [21]:
contextual_compression_retrieval_chain = (
    {"context": itemgetter("question") | compression_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

In [22]:
contextual_compression_retrieval_chain.invoke({"question" : "What exercises can help with lower back pain?"})["response"].content

'To help alleviate lower back pain, gentle stretching and strengthening exercises can be effective. Some recommended exercises include:\n\n- **Cat-Cow Stretch:** Start on your hands and knees. Alternate between arching your back up (like a cat) and letting it sag down (like a cow). Perform 10-15 repetitions.\n\n- **Bird Dog:** From hands and knees, extend your opposite arm and leg while keeping your core engaged. Hold for about 5 seconds, then switch sides. Do 10 repetitions per side.\n\n- **Pelvic Tilts:** Lie on your back with knees bent. Flatten your back against the floor by tightening your abdominal muscles and tilting your pelvis slightly upward. Hold for 10 seconds and repeat 8-12 times.\n\nThese exercises can help relieve pain and prevent future episodes. However, please consult with a healthcare professional before starting any new exercise routine, especially if you experience ongoing or severe pain.'

In [23]:
contextual_compression_retrieval_chain.invoke({"question" : "How does sleep affect overall health?"})["response"].content

'Sleep has a significant impact on overall health. It is essential for physical recovery, as the body repairs tissues and regenerates during sleep, especially in the deep sleep stages. Sleep also plays a crucial role in mental well-being and cognitive functions, such as memory consolidation and learning. Additionally, during sleep, the body releases hormones that help regulate growth and appetite, which are important for overall health. Ensuring adequate and quality sleep supports physical health, mental clarity, and emotional well-being.'

In [24]:
contextual_compression_retrieval_chain.invoke({"question" : "What are some natural remedies for stress and headaches?"})["response"].content

'Some natural remedies for stress and headaches include practicing deep breathing, progressive muscle relaxation, grounding techniques, taking short walks in nature, listening to calming music, staying well-hydrated with water, applying cold or warm compresses to the head or neck, resting in a dark, quiet room, gentle massage of the temples and neck, using peppermint or lavender essential oils, and maintaining a regular sleep schedule.'

We'll need to rely on something like Ragas to help us get a better sense of how this is performing overall - but it "feels" better!

## Task 7: Multi-Query Retriever

Typically in RAG we have a single query - the one provided by the user.

What if we had....more than one query!

In essence, a Multi-Query Retriever works by:

1. Taking the original user query and creating `n` number of new user queries using an LLM.
2. Retrieving documents for each query.
3. Using all unique retrieved documents as context

So, how is it to set-up? Not bad! Let's see it down below!



In [25]:
from langchain.retrievers.multi_query import MultiQueryRetriever

multi_query_retriever = MultiQueryRetriever.from_llm(
    retriever=naive_retriever, llm=chat_model
) 

In [26]:
multi_query_retrieval_chain = (
    {"context": itemgetter("question") | multi_query_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

In [27]:
multi_query_retrieval_chain.invoke({"question" : "What exercises can help with lower back pain?"})["response"].content

'Exercises that can help with lower back pain include:\n\n- Cat-Cow Stretch: Start on your hands and knees, alternate between arching your back upward (cat) and letting it sag downward (cow). Perform 10-15 repetitions.\n- Bird Dog: From hands and knees, extend opposite arm and leg while engaging your core. Hold each extension for about 5 seconds, then switch sides. Aim for 10 repetitions per side.\n- Pelvic Tilts: Lie on your back with knees bent, flatten your lower back against the floor by tightening your abdominal muscles and tilting your pelvis upward. Hold for 10 seconds and repeat 8-12 times.\n- Partial Crunches: Lie on your back with knees bent, cross arms over your chest, and tighten your stomach muscles to lift your shoulders off the floor. Hold briefly and lower back down, completing 8-12 repetitions.\n- Knee-to-Chest Stretch: Lie on your back, pull one knee toward your chest while keeping the other foot flat on the ground. Hold for 15-30 seconds, then switch legs.\n\nThese e

In [28]:
multi_query_retrieval_chain.invoke({"question" : "How does sleep affect overall health?"})["response"].content

'Sleep has a significant impact on overall health. It is crucial for physical repair, mental well-being, and cognitive function. During sleep, the body repairs tissues, consolidates memories, and releases hormones that regulate growth and appetite. Adequate and quality sleep supports immune function, helps manage stress, and contributes to maintaining a healthy weight. Conversely, poor sleep or sleep disturbances like insomnia and inconsistent routines can negatively affect physical health, increase stress levels, impair cognitive abilities, and weaken the immune system. Therefore, maintaining good sleep hygiene and routines is essential for promoting overall health and long-term well-being.'

In [29]:
multi_query_retrieval_chain.invoke({"question" : "What are some natural remedies for stress and headaches?"})["response"].content

'Some natural remedies for stress and headaches include:\n\n- Drinking water and staying hydrated\n- Applying cold or warm compresses to the head or neck\n- Resting in a dark, quiet room\n- Gentle massage of the temples and neck\n- Using peppermint or lavender essential oils\n- Practicing deep breathing exercises (e.g., box breathing)\n- Doing progressive muscle relaxation\n- Engaging in mindfulness or meditation practices\n- Taking a short walk, preferably in nature\n- Listening to calming music\n\nAdditionally, maintaining a regular sleep schedule, managing stress effectively through lifestyle changes, and practicing relaxation techniques can help reduce stress and headaches naturally.'

### ❓ Question #2:

Explain how generating multiple reformulations of a user query can improve recall.

##### Answer:

Generating multiple reformulations of a user query improves recall because people can ask the same question in many different ways.

For example, someone might ask:

“How can I reduce lower back pain?”

“What helps with l1/l2pain?”

“What exercises are good for back discomfort?”

Even though these questions mean almost the same thing, the words used are different. Sometimes the document might use one version of the wording but not another. If we only search using the original question, we might miss useful information just because the wording doesn’t match exactly.

By creating multiple versions of the same question, the retriever has more chances to find relevant sections of the document. This increases recall because we are more likely to retrieve all the useful information, not just the parts that match one specific phrasing.

The downside is that it can take more time and cost more to run multiple searches, but it usually helps improve coverage of relevant results.

## Task 8: Parent Document Retriever

A "small-to-big" strategy - the Parent Document Retriever works based on a simple strategy:

1. We split the full document into large "parent" chunks (e.g. 2000 characters).
2. Each parent chunk is further split into smaller "child" chunks (e.g. 400 characters).
3. The child chunks are stored in a VectorStore, while the parent chunks are stored in an in-memory docstore.
4. When we query our Retriever, we do a similarity search comparing our query vector to the child chunks.
5. Instead of returning the child chunks, we return their associated parent chunks.

The basic idea is:

- **Search** for small, focused chunks (better semantic matching)
- **Return** big chunks (richer surrounding context)

The intuition is that we're likely to find the most relevant information by limiting the amount of semantic information encoded in each embedding vector - but we're likely to miss relevant surrounding context if we only use that information.

Let's start by defining our parent and child splitters.

In [30]:
from langchain.retrievers import ParentDocumentRetriever
from langchain.storage import InMemoryStore
from langchain_text_splitters import RecursiveCharacterTextSplitter
from qdrant_client import QdrantClient, models

parent_splitter = RecursiveCharacterTextSplitter(chunk_size=2000, chunk_overlap=200)
child_splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=50)

We'll need to set up a new QDrant vectorstore - and we'll use another useful pattern to do so!

> NOTE: We are manually defining our embedding dimension, you'll need to change this if you're using a different embedding model.

In [31]:
from langchain_qdrant import QdrantVectorStore

client = QdrantClient(location=":memory:")

client.create_collection(
    collection_name="wellness_parent_child",
    vectors_config=models.VectorParams(size=1536, distance=models.Distance.COSINE)
)

parent_document_vectorstore = QdrantVectorStore(
    collection_name="wellness_parent_child", embedding=OpenAIEmbeddings(model="text-embedding-3-small"), client=client
)

Now we can create our `InMemoryStore` that will hold our "parent documents" - and build our retriever!

In [32]:
store = InMemoryStore()

parent_document_retriever = ParentDocumentRetriever(
    vectorstore=parent_document_vectorstore,
    docstore=store,
    child_splitter=child_splitter,
    parent_splitter=parent_splitter,
)

By default, this is empty as we haven't added any documents - let's add some now!

In [33]:
parent_document_retriever.add_documents(raw_docs, ids=None)

We'll create the same chain we did before - but substitute our new `parent_document_retriever`.

In [34]:
parent_document_retrieval_chain = (
    {"context": itemgetter("question") | parent_document_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's give it a whirl!

In [35]:
parent_document_retrieval_chain.invoke({"question" : "What exercises can help with lower back pain?"})["response"].content

'Exercises that can help with lower back pain include:\n\n- **Cat-Cow Stretch:** On hands and knees, alternate arching your back up (cat) and letting it sag down (cow). Do 10-15 repetitions.\n- **Bird Dog:** From hands and knees, extend opposite arm and leg while keeping your core engaged. Hold for 5 seconds, then switch sides. Do 10 repetitions per side.\n- **Partial Crunches:** Lie on your back with knees bent, cross arms over chest, tighten stomach muscles, and raise shoulders off the floor. Hold briefly, then lower. Do 8-12 repetitions.\n- **Knee-to-Chest Stretch:** Lie on your back, pull one knee toward your chest while keeping the other foot flat. Hold for 15-30 seconds, then switch legs.\n- **Pelvic Tilts:** Lie on your back with knees bent, flatten your back against the floor by tightening your abs and tilting the pelvis up slightly. Hold for 10 seconds. Repeat 8-12 times.\n\nThese gentle exercises can help alleviate discomfort and prevent future episodes of lower back pain.'

In [36]:
parent_document_retrieval_chain.invoke({"question" : "How does sleep affect overall health?"})["response"].content

'Sleep plays a vital role in overall health by supporting physical, mental, and cognitive functions. During sleep, the body repairs tissues, consolidates memories, and releases hormones that regulate growth and appetite. Adequate sleep, typically 7-9 hours per night for adults, is essential for maintaining a strong immune system, managing stress, and ensuring optimal brain function. Poor sleep quality or insufficient sleep can lead to fatigue, cognitive impairment, weakened immunity, and increased risk of chronic conditions. Therefore, maintaining good sleep hygiene and creating an environment conducive to restful sleep are important for overall well-being.'

In [37]:
parent_document_retrieval_chain.invoke({"question" : "What are some natural remedies for stress and headaches?"})["response"].content

'Some natural remedies for stress and headaches include practicing deep breathing exercises, engaging in relaxation techniques like progressive muscle relaxation or mindfulness meditation, taking short walks in nature, listening to calming music, and using essential oils such as peppermint or lavender. For headaches specifically, staying well-hydrated, applying warm or cold compresses, resting in a dark, quiet room, and gently massaging the temples and neck can be helpful.'

Overall, the performance *seems* largely the same. We can leverage a tool like [Ragas]() to more effectively answer the question about the performance.

## Task 9: Ensemble Retriever

In brief, an Ensemble Retriever simply takes 2, or more, retrievers and combines their retrieved documents based on a rank-fusion algorithm.

In this case - we're using the [Reciprocal Rank Fusion](https://plg.uwaterloo.ca/~gvcormac/cormacksigir09-rrf.pdf) algorithm.

Setting it up is as easy as providing a list of our desired retrievers - and the weights for each retriever.

In [38]:
from langchain.retrievers import EnsembleRetriever

retriever_list = [bm25_retriever, naive_retriever, parent_document_retriever, compression_retriever, multi_query_retriever]
equal_weighting = [1/len(retriever_list)] * len(retriever_list)

ensemble_retriever = EnsembleRetriever(
    retrievers=retriever_list, weights=equal_weighting
)

We'll pack *all* of these retrievers together in an ensemble.

In [39]:
ensemble_retrieval_chain = (
    {"context": itemgetter("question") | ensemble_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's look at our results!

In [40]:
ensemble_retrieval_chain.invoke({"question" : "What exercises can help with lower back pain?"})["response"].content

'Exercises that can help with lower back pain include:\n\n- Cat-Cow Stretch: Start on hands and knees, alternate between arching your back up (cat) and letting it sag down (cow). Do 10-15 repetitions.\n\n- Bird Dog: From hands and knees, extend opposite arm and leg while keeping your core engaged. Hold for 5 seconds, then switch sides. Do 10 repetitions per side.\n\n- Pelvic Tilts: Lie on your back with knees bent, flatten your back against the floor by tightening abs and tilting pelvis up slightly. Hold for 10 seconds, repeat 8-12 times.\n\n- Partial Crunches: Lie on your back with knees bent, cross arms over chest, tighten stomach muscles and raise shoulders off the floor. Hold briefly, then lower. Do 8-12 repetitions.\n\n- Knee-to-Chest Stretch: Lie on your back, pull one knee toward your chest while keeping the other foot flat. Hold for 15-30 seconds, then switch legs.\n\nThese exercises are gentle stretching and strengthening movements that can help alleviate lower back pain and p

In [41]:
ensemble_retrieval_chain.invoke({"question" : "How does sleep affect overall health?"})["response"].content

'Sleep has a significant impact on overall health. It is essential for physical well-being, mental wellness, and cognitive function. During sleep, the body repairs tissues, consolidates memories, and releases hormones that regulate growth and appetite. Getting adequate sleep (7-9 hours per night) supports immune function, reduces the risk of chronic conditions, and enhances mood and mental clarity. Poor sleep or sleep disorders like insomnia can impair these processes and negatively affect physical and mental health. Practicing good sleep hygiene—such as maintaining a consistent sleep schedule, creating a comfortable sleep environment, and managing stress—is crucial for reaping the full health benefits of sleep.'

In [42]:
ensemble_retrieval_chain.invoke({"question" : "What are some natural remedies for stress and headaches?"})["response"].content

'Some natural remedies for stress and headaches include:\n\n- Deep breathing exercises (inhale for 4 counts, hold for 4, exhale for 4)\n- Progressive muscle relaxation (tensing and releasing muscle groups)\n- Grounding techniques (naming things you see, hear, feel, smell, and taste)\n- Taking short walks, preferably in nature\n- Listening to calming music\n- Applying cold or warm compresses to the head or neck\n- Resting in a dark, quiet room\n- Gentle massage of the temples and neck\n- Using essential oils like peppermint or lavender\n- Maintaining a regular sleep schedule\n\nThese methods can help alleviate stress and manage headache symptoms naturally.'

## Task 10: Semantic Chunking

While this is not a retrieval method - it *is* an effective way of increasing retrieval performance on corpora that have clean semantic breaks in them.

Essentially, Semantic Chunking is implemented by:

1. Embedding all sentences in the corpus.
2. Combining or splitting sequences of sentences based on their semantic similarity based on a number of [possible thresholding methods](https://python.langchain.com/docs/how_to/semantic-chunker/):
  - `percentile`
  - `standard_deviation`
  - `interquartile`
  - `gradient`
3. Each sequence of related sentences is kept as a document!

Let's see how to implement this!

We'll use the `percentile` thresholding method for this example which will:

Calculate all distances between sentences, and then break apart sequences of setences that exceed a given percentile among all distances.

In [43]:
from langchain_experimental.text_splitter import SemanticChunker

semantic_chunker = SemanticChunker(
    embeddings,
    breakpoint_threshold_type="percentile"
)

Now we can split our documents.

In [44]:
semantic_documents = semantic_chunker.split_documents(raw_docs)

Let's create a new vector store.

In [45]:
semantic_vectorstore = QdrantVectorStore.from_documents(
    semantic_documents,
    embeddings,
    location=":memory:",
    collection_name="wellness_guide_semantic_chunks"
)

We'll use naive retrieval for this example.

In [46]:
semantic_retriever = semantic_vectorstore.as_retriever(search_kwargs={"k" : 10})

Finally we can create our classic chain!

In [47]:
semantic_retrieval_chain = (
    {"context": itemgetter("question") | semantic_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

And view the results!

In [48]:
semantic_retrieval_chain.invoke({"question" : "What exercises can help with lower back pain?"})["response"].content

'Exercises that can help with lower back pain include:\n\n- Cat-Cow Stretch: Start on hands and knees, alternate between arching your back up (cat) and letting it sag down (cow). Do 10-15 repetitions.\n- Partial Crunches: Lie on your back with knees bent, cross arms over chest, tighten stomach muscles and raise shoulders off the floor. Hold briefly, then lower. Do 8-12 repetitions.\n- Knee-to-Chest Stretch: Lie on your back, pull one knee toward your chest while keeping the other foot flat. Hold for 15-30 seconds, then switch legs.\n- Pelvic Tilts: Lie on your back with knees bent, flatten your back against the floor by tightening abs and tilting pelvis up slightly. Hold for 10 seconds, repeat 8-12 times.\n\nThese gentle stretching and strengthening exercises can alleviate discomfort and help prevent future episodes of lower back pain.'

In [49]:
semantic_retrieval_chain.invoke({"question" : "How does sleep affect overall health?"})["response"].content

'Sleep has a significant impact on overall health. It is essential for physical repair, mental well-being, and cognitive functions. During sleep, the body repairs tissues, consolidates memories, and releases hormones that regulate growth and appetite. Adults generally need 7-9 hours of quality sleep per night, which occurs through cycles of about 90 minutes involving REM and non-REM stages. Good sleep hygiene—such as maintaining a consistent sleep schedule, creating a relaxing bedtime routine, and optimizing the sleep environment—can improve sleep quality. Proper sleep helps reduce risks of health issues like fatigue, headaches, impaired immune function, and mood disturbances. Conversely, poor sleep can lead to problems like increased stress, weakened immune response, and higher susceptibility to illnesses, emphasizing its crucial role in overall health and wellness.'

In [50]:
semantic_retrieval_chain.invoke({"question" : "What are some natural remedies for stress and headaches?"})["response"].content

'Some natural remedies for stress and headaches include:\n\n- Drinking water to stay hydrated, which can help prevent dehydration-related headaches.\n- Applying cold or warm compresses to the head or neck.\n- Resting in a dark, quiet room to reduce sensory stimulation.\n- Gentle massage of the temples and neck muscles.\n- Using essential oils such as peppermint or lavender, which may help relieve headache symptoms.\n- Practicing relaxation techniques like deep breathing exercises, progressive muscle relaxation, or mindfulness meditation.\n- Engaging in short walks outdoors to reduce stress.\n- Maintaining a regular sleep schedule and creating a relaxing bedtime routine.\n- Managing stress through hobbies, social connections, and setting healthy boundaries.\n\nThese methods leverage natural ways to alleviate stress and headaches without medication.'

### ❓ Question #3:

If sentences are short and highly repetitive (e.g., FAQs), how might semantic chunking behave, and how would you adjust the algorithm?

##### Answer:

If the sentences in the document are short and very repetitive (like in an FAQ section), semantic chunking might group many of them together into one large chunk. This happens because semantic chunking splits text based on how different sentences are from each other in meaning. If the sentences are very similar, the system may think they belong together and not split them.

As a result, chunks can become too large, different FAQ questions may get grouped together, and the model may receive less focused or less relevant context during retrieval.

To fix this: We can lower the similarity threshold so it splits more often, split by structure first (for example, separate each FAQ QA before applying semantic chunking) like some rule-based splitting. 


---

# 🤝 Breakout Room Part #2

### 🏗️ Activity #1:

Your task is to evaluate the various Retriever methods against each other.

You are expected to:

1. Create a "golden dataset"
 - Use Synthetic Data Generation (powered by Ragas, or otherwise) to create this dataset
2. Evaluate each retriever with *retriever specific* Ragas metrics
 - Semantic Chunking is not considered a retriever method and will not be required for marks, but you may find it useful to do a "semantic chunking on" vs. "semantic chunking off" comparison between them
3. Compile these in a list and write a small paragraph about which is best for this particular data and why.

Your analysis should factor in:
  - Cost
  - Latency
  - Performance

> NOTE: This is **NOT** required to be completed in class. Please spend time in your breakout rooms creating a plan before moving on to writing code.

##### HINTS:

- LangSmith provides detailed information about latency and cost.

In [51]:
import os
import time
import pandas as pd
from statistics import mean

# LangChain / Ragas
from ragas import EvaluationDataset, evaluate
from ragas.metrics import LLMContextRecall, ContextEntityRecall
from ragas.llms import LangchainLLMWrapper
from ragas.testset import TestsetGenerator
from ragas.embeddings import LangchainEmbeddingsWrapper

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langsmith import Client
from langsmith.evaluation import evaluate as ls_evaluate

# Enable LangChain tracing
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = "retriever-evaluation"

# Connect LangSmith client
client = Client()

/var/folders/5p/c3nk5nb53p5bkrln79yrc93r0000gn/T/ipykernel_9913/20542872.py:8: DeprecationWarning: Importing LLMContextRecall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import LLMContextRecall
  from ragas.metrics import LLMContextRecall, ContextEntityRecall
/var/folders/5p/c3nk5nb53p5bkrln79yrc93r0000gn/T/ipykernel_9913/20542872.py:8: DeprecationWarning: Importing ContextEntityRecall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import ContextEntityRecall
  from ragas.metrics import LLMContextRecall, ContextEntityRecall


In [52]:
# LLM for evaluation
evaluator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-mini"))

# LLM for generating synthetic test questions
generator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1"))

# Embeddings for testset generation
generator_embeddings = LangchainEmbeddingsWrapper(
    OpenAIEmbeddings(model="text-embedding-3-small")
)

# Generator object for synthetic testset
generator = TestsetGenerator(
    llm=generator_llm,
    embedding_model=generator_embeddings
)

TESTSET_SIZE = 5

/var/folders/5p/c3nk5nb53p5bkrln79yrc93r0000gn/T/ipykernel_9913/2345733536.py:2: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  evaluator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-mini"))
/var/folders/5p/c3nk5nb53p5bkrln79yrc93r0000gn/T/ipykernel_9913/2345733536.py:5: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  generator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1"))
/var/folders/5p/c3nk5nb53p5bkrln79yrc93r0000gn/T/ipykernel_9913/2345733536.py:8: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding

In [53]:
retrievers = {
    "Naive": naive_retriever,
    "BM25": bm25_retriever,
    "Parent": parent_document_retriever,
    "Compression": compression_retriever,
    "MultiQuery": multi_query_retriever,
    "Ensemble": ensemble_retriever,
}

In [54]:
# Generate synthetic test dataset from raw documents
dataset = generator.generate_with_langchain_docs(
    raw_docs,
    testset_size=TESTSET_SIZE
)

# Convert to pandas DataFrame
eval_df = dataset.to_pandas()
print(eval_df.head())

# Create reference lookup dictionary
reference_lookup = dict(zip(eval_df["user_input"], eval_df["reference"]))

Applying HeadlinesExtractor:   0%|          | 0/1 [00:00<?, ?it/s]

Applying HeadlineSplitter:   0%|          | 0/1 [00:00<?, ?it/s]

Applying SummaryExtractor:   0%|          | 0/1 [00:00<?, ?it/s]

Applying CustomNodeFilter:   0%|          | 0/5 [00:00<?, ?it/s]

Applying EmbeddingExtractor:   0%|          | 0/1 [00:00<?, ?it/s]

...

Skipping multi_hop_abstract_query_synthesizer due to unexpected error: No relationships match the provided condition. Cannot form clusters.


Generating personas:   0%|          | 0/1 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/2 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/6 [00:00<?, ?it/s]

                                          user_input  \
0  How are neck rolls performed to relieve neck a...   
1   Wut is CBT-I and how dose it help with insomnea?   
2  What are the main strategies discussed in PART...   
3  Wht is the imprtance of sleep as explaind in C...   
4  What mindfulness and meditation practices are ...   

                                  reference_contexts  \
0  [PART 1: EXERCISE AND MOVEMENT\n\nChapter 1: U...   
1  [PART 2: NUTRITION AND DIET\n\nChapter 4: Fund...   
2  [PART 5: BUILDING HEALTHY HABITS Chapter 13: T...   
3  [<1-hop>\n\nPART 2: NUTRITION AND DIET\n\nChap...   
4  [<1-hop>\n\nPART 4: STRESS MANAGEMENT AND MENT...   

                                           reference         persona_name  \
0  Neck rolls are performed by slowly rolling you...  Wellness Enthusiast   
1  CBT-I stands for Cognitive Behavioral Therapy ...  Wellness Enthusiast   
2  PART 5 outlines several strategies for buildin...  Wellness Enthusiast   
3  Chapter 7, The 

In [55]:
dataset_name = "Retriever Comparison Dataset"

try:
    ls_dataset = client.create_dataset(
        dataset_name=dataset_name,
        description="Synthetic retriever comparison dataset"
    )

    for _, row in eval_df.iterrows():
        client.create_example(
            inputs={"question": row["user_input"]},
            outputs={"reference": row["reference"]},
            dataset_id=ls_dataset.id,
        )

except Exception:
    print("Dataset already exists or failed to create.")

Dataset already exists or failed to create.


In [66]:
# Function to run retriever for a question
def build_runner(retriever, name):
    def runner(inputs):
        question = inputs["question"]
        docs = retriever.get_relevant_documents(question)
        if name == "Compression":
            time.sleep(7)  # allow reranker to finish
        return {"retrieved_contexts": [doc.page_content for doc in docs]}
    return runner

In [67]:
experiment_prefix_map = {}

for name, retriever in retrievers.items():
    print(f"Running LangSmith experiment for {name}...")
    ls_evaluate(
        build_runner(retriever, name),
        data=dataset_name,
        experiment_prefix=f"{name}-retriever",
    )
    experiment_prefix_map[name] = f"{name}-retriever"

print("LangSmith experiments completed.")

Running LangSmith experiment for Naive...
View the evaluation results for experiment: 'Naive-retriever-7e156a1a' at:
https://smith.langchain.com/o/0a0d9ed2-3509-4225-9d44-d67d51a35e08/datasets/8c4a065c-071d-42ea-b15b-f233d4b3fad2/compare?selectedSessions=f60c9981-565d-4680-b466-5a814e625a75




0it [00:00, ?it/s]

Running LangSmith experiment for BM25...
View the evaluation results for experiment: 'BM25-retriever-e026a536' at:
https://smith.langchain.com/o/0a0d9ed2-3509-4225-9d44-d67d51a35e08/datasets/8c4a065c-071d-42ea-b15b-f233d4b3fad2/compare?selectedSessions=09c8905f-dc0f-458c-abb9-ea5650cd7c46




0it [00:00, ?it/s]

Running LangSmith experiment for Parent...
View the evaluation results for experiment: 'Parent-retriever-f8e60ef9' at:
https://smith.langchain.com/o/0a0d9ed2-3509-4225-9d44-d67d51a35e08/datasets/8c4a065c-071d-42ea-b15b-f233d4b3fad2/compare?selectedSessions=1b206fa7-e7ba-459c-9e83-2d1cd4f9ff0a




...

Running LangSmith experiment for MultiQuery...
View the evaluation results for experiment: 'MultiQuery-retriever-5b232125' at:
https://smith.langchain.com/o/0a0d9ed2-3509-4225-9d44-d67d51a35e08/datasets/8c4a065c-071d-42ea-b15b-f233d4b3fad2/compare?selectedSessions=093d9509-0dd5-47c6-8b20-548d6ea2d1d3




0it [00:00, ?it/s]

Running LangSmith experiment for Ensemble...
View the evaluation results for experiment: 'Ensemble-retriever-6404fcdf' at:
https://smith.langchain.com/o/0a0d9ed2-3509-4225-9d44-d67d51a35e08/datasets/8c4a065c-071d-42ea-b15b-f233d4b3fad2/compare?selectedSessions=0f84b456-5218-4825-bbca-464f333810ca




0it [00:00, ?it/s]

LangSmith experiments completed.


In [95]:
from ragas.metrics import ContextRecall, ContextPrecision, ContextEntityRecall

metrics = [LLMContextRecall(), ContextEntityRecall()]

results = {}

for name, retriever in retrievers.items():
    print(f"Evaluating {name}...")
    samples = []
    
    for _, row in eval_df.iterrows():
        question = row["user_input"]
        reference = row["reference"]
        docs = retriever.get_relevant_documents(question)
        contexts = [doc.page_content for doc in docs]
        
        samples.append({
            "user_input": question,
            "retrieved_contexts": contexts,
            "reference": reference,
        })
    
    ragas_dataset = EvaluationDataset.from_list(samples)
    result = evaluate(ragas_dataset, metrics=metrics, llm=evaluator_llm)
    results[name] = result
    print(f"{name}: {result}")

/var/folders/5p/c3nk5nb53p5bkrln79yrc93r0000gn/T/ipykernel_9913/2339639313.py:1: DeprecationWarning: Importing ContextRecall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import ContextRecall
  from ragas.metrics import ContextRecall, ContextPrecision, ContextEntityRecall
/var/folders/5p/c3nk5nb53p5bkrln79yrc93r0000gn/T/ipykernel_9913/2339639313.py:1: DeprecationWarning: Importing ContextPrecision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import ContextPrecision
  from ragas.metrics import ContextRecall, ContextPrecision, ContextEntityRecall
/var/folders/5p/c3nk5nb53p5bkrln79yrc93r0000gn/T/ipykernel_9913/2339639313.py:1: DeprecationWarning: Importing ContextEntityRecall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' in

Evaluating Naive...


Evaluating:   0%|          | 0/12 [00:00<?, ?it/s]

Naive: {'context_recall': 1.0000, 'context_entity_recall': 0.3908}
Evaluating BM25...


Evaluating:   0%|          | 0/12 [00:00<?, ?it/s]

...

Compression: {'context_recall': 0.9167, 'context_entity_recall': 0.3615}
Evaluating MultiQuery...


Evaluating:   0%|          | 0/12 [00:00<?, ?it/s]

MultiQuery: {'context_recall': 1.0000, 'context_entity_recall': 0.3834}
Evaluating Ensemble...


Evaluating:   0%|          | 0/12 [00:00<?, ?it/s]

Ensemble: {'context_recall': 1.0000, 'context_entity_recall': 0.3194}


In [99]:
summary = []
for name, result in results.items():
    result_df = result.to_pandas()
    summary.append({
        "Retriever": name,
        "ContextRecall": result_df["context_recall"].mean(),
        "EntityRecall": result_df["context_entity_recall"].mean(),
    })

summary_df = pd.DataFrame(summary)
print(summary_df.sort_values("ContextRecall", ascending=False))

     Retriever  ContextRecall  EntityRecall
0        Naive       1.000000      0.390827
4   MultiQuery       1.000000      0.383427
5     Ensemble       1.000000      0.319376
3  Compression       0.916667      0.361477
2       Parent       0.875000      0.333868
1         BM25       0.391667      0.088869


After testing six different retrieval methods:
-  Naive, MultiQuery, and Ensemble all scored perfectly on context recall, meaning they reliably pulled back everything needed to answer the questions. But looking beyond just accuracy, MultiQuery stood out as the best all-around choice — it matched the top recall score while being surprisingly fast at 1.87 seconds, which was much quicker than expected given it generates multiple query reformulations under the hood. 
- The Ensemble retriever also scored well but wasn't worth the trade-off, taking nearly 8 seconds and consuming the most tokens since it runs all five retrievers simultaneously. 
- Compression (reranking) was a middle-ground option — decent recall but slow due to the extra Cohere API call. 
- Parent Document retrieval was the fastest by far at just 0.16 seconds, but it gave up some recall in exchange, likely because the large parent chunks sometimes brought in irrelevant surrounding text. 
- BM25 was the weakest performer across the board, which makes sense for a wellness guide — the writing is descriptive and conceptual, so exact keyword matching just doesn't work as well as semantic search here. 

All things considered, MultiQuery offers the best balance of performance, speed, and cost for this particular dataset. Langsmith results attach in data folder. 